<a href="https://colab.research.google.com/github/sokrypton/7.571/blob/main/L6/sequence_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to Sequence Analysis & Probabilities

**A hands-on tutorial using only Python and NumPy**

We're going to answer one deceptively simple question:

> **"If two protein sequences look similar, is that meaningful — or just coincidence?"**

By the end of this notebook you'll understand:
1. How to build a **null distribution** and compute a **p-value** from scratch
2. How **substitution matrices** (like BLOSUM) are derived from real evolutionary data
3. How **Smith-Waterman** local alignment works
4. How **E-values** correct for multiple comparisons when searching a whole genome

---
## Part 0 — Setup

We only need NumPy and matplotlib. Everything else we build ourselves.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import urllib.request
import io, gzip, re, textwrap, time

np.random.seed(42)

# Standard amino acid alphabet (20 standard + gap)
AA = 'ACDEFGHIKLMNPQRSTVWY'
AA_SET = set(AA)
AA_INDEX = {a: i for i, a in enumerate(AA)}

print(f"Amino acid alphabet ({len(AA)} residues): {AA}")

---
## Part 1 — Are These Sequences Similar by Chance?

In [ ]:
# Two protein fragments — an oxidoreductase and a possible distant homolog
# They share ~28% identity. Is that meaningful, or just noise?
seq_a = "MRIIVALITGATGQIGRFAIAQLLAQCEVLGIDTATSEQVARQCVDAMKPGGTFYTCARVADRDQSFAAALQASLKAFGRID"
seq_b = "DRIASTNAVRETQIICSGYNAQSLLLSFSLLADSCCTEDKGKRCFNSKTRLGTAALGAFVALADSTQGHTLQAELGAFPKVC"

# Simple percent identity for two equal-length sequences
def percent_identity(s1, s2):
    """Fraction of positions where two sequences have the same residue."""
    assert len(s1) == len(s2), "Sequences must be same length"
    matches = sum(a == b for a, b in zip(s1, s2))
    return matches / len(s1)

obs_identity = percent_identity(seq_a, seq_b)
print(f"Sequence A: {seq_a}")
print(f"Sequence B: {seq_b}")
print(f"Length:     {len(seq_a)}")
print(f"Identity:   {obs_identity:.1%}")
print()

# Show position-by-position comparison
midline = ''.join('|' if a == b else ' ' for a, b in zip(seq_a, seq_b))
print(f"A: {seq_a}")
print(f"   {midline}")
print(f"B: {seq_b}")
print(f"\n~28% identity — is that real, or could you get that by chance?")

### Is this identity significant?

**Null hypothesis:** The two sequences are unrelated; any similarity is due to chance alone.

**Strategy:** Generate thousands of **random protein sequences** drawn from realistic amino acid frequencies (matching E. coli proteome composition) and measure their identity. This models what happens when two completely unrelated proteins are compared.

In [ ]:
# Amino acid frequencies from the E. coli K-12 proteome
# These are realistic background frequencies — some amino acids (L, A, G) are common,
# others (W, C) are rare. This matters: if both sequences are rich in leucine,
# random matches at leucine positions are more likely.
AA_FREQ = np.array([
    0.087,  # A - Alanine
    0.033,  # C - Cysteine
    0.047,  # D - Aspartate
    0.050,  # E - Glutamate
    0.040,  # F - Phenylalanine
    0.089,  # G - Glycine
    0.034,  # H - Histidine
    0.037,  # I - Isoleucine
    0.081,  # K - Lysine
    0.085,  # L - Leucine
    0.015,  # M - Methionine
    0.040,  # N - Asparagine
    0.051,  # P - Proline
    0.038,  # Q - Glutamine
    0.041,  # R - Arginine
    0.070,  # S - Serine
    0.058,  # T - Threonine
    0.065,  # V - Valine
    0.010,  # W - Tryptophan
    0.030,  # Y - Tyrosine
])
AA_FREQ = AA_FREQ / AA_FREQ.sum()  # normalize

def random_protein(length, aa_freq=AA_FREQ):
    """Generate a random protein sequence with realistic amino acid composition."""
    return ''.join(np.random.choice(list(AA), size=length, p=aa_freq))

def random_proteins_matrix(n_seqs, length, aa_freq=AA_FREQ):
    """Generate n random protein sequences as a (n, length) integer array. Much faster than strings."""
    return np.random.choice(len(AA), size=(n_seqs, length), p=aa_freq)

# What identity do we expect by pure chance?
# Theoretical: sum of p_i^2 (probability both positions have the same amino acid)
expected_random_identity = np.sum(AA_FREQ ** 2)
print(f"Theoretical random identity (sum of p_i²): {expected_random_identity:.1%}")
print(f"  (With uniform frequencies over 20 AAs it would be {1/20:.1%})")
print(f"  (Biased composition pushes it higher — common AAs match more often)")
print()

# Empirical check — vectorized with numpy (no Python loops over positions)
n_trials = 100_000
L = len(seq_a)  # 82

seqs_a = random_proteins_matrix(n_trials, L)
seqs_b = random_proteins_matrix(n_trials, L)
null_identities = np.mean(seqs_a == seqs_b, axis=1)  # fraction matching per row

print(f"Empirical null identity (L={L}): {null_identities.mean():.1%} ± {null_identities.std():.1%}")
print(f"Observed identity:               {obs_identity:.1%}")
print()

# Compute empirical p-value
p_value = np.mean(null_identities >= obs_identity)
print(f"Empirical p-value: {p_value}")

In [ ]:
# Visualize the null distribution and overlay the theoretical Binomial distribution
# Each position is an independent trial: does the same AA land in both sequences?
# Probability of a match at one position: p_match = sum(p_i^2)
# Number of matches in L positions ~ Binomial(L, p_match)
# So percent identity = Binomial(L, p_match) / L

from scipy.stats import binom

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(null_identities, bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='white',
        label='Empirical (50k random pairs)')

# Binomial PMF (number of matches k = 0..L, converted to fraction k/L)
p_match = expected_random_identity  # sum of p_i^2
k_vals = np.arange(0, L + 1)
binom_pmf = binom.pmf(k_vals, L, p_match)
# Convert to density over percent identity (scale by L since identity = k/L)
ax.plot(k_vals / L, binom_pmf * L, 'k-', linewidth=2, label=f'Binomial(L={L}, p={p_match:.3f})')

ax.axvline(obs_identity, color='red', linewidth=2, linestyle='--', label=f'Observed: {obs_identity:.1%}')
ax.axvline(p_match, color='gray', linewidth=1, linestyle='-', label=f'Expected: {p_match:.1%}')

ax.set_xlabel('Percent Identity', fontsize=13)
ax.set_ylabel('Density', fontsize=13)
ax.set_title(f'Null Distribution of Sequence Identity (L = {L})\nEach position is a Bernoulli trial with p = Σpᵢ² = {p_match:.3f}', fontsize=14)
ax.legend(fontsize=11)

# Shade the p-value region
ax.axvspan(obs_identity, 1.0, alpha=0.15, color='red')
p_str = f'p = {p_value:.1e}' if p_value > 0 else f'p < {1/n_trials:.1e}'
ax.text(obs_identity + 0.005, ax.get_ylim()[1] * 0.8, p_str, fontsize=14, color='red', fontweight='bold')

plt.tight_layout()
plt.show()

# Also compute the exact binomial p-value
p_value_binom = 1 - binom.cdf(int(obs_identity * L) - 1, L, p_match)
print(f"Exact binomial p-value: {p_value_binom:.2e}")
print(f"Empirical p-value:      {p_value if p_value > 0 else f'< {1/n_trials:.1e}'}")

### How does sequence length affect what counts as "significant"?

This is a crucial insight: **the same percent identity means very different things at different lengths.**

Since each position is an independent Bernoulli trial with success probability $p = \sum p_i^2$, the number of matching positions follows a **Binomial distribution**: $\text{matches} \sim \text{Binomial}(L, p)$.

As $L$ grows, the binomial gets tighter relative to its mean (standard deviation scales as $\sqrt{L}$, but the mean scales as $L$). So the bar for significance drops with length.

Let's compute the **identity threshold for p < 0.05** across a range of lengths — both empirically and from the exact binomial.

In [ ]:
# For each sequence length, find the identity threshold where p = 0.05
# We can now do this EXACTLY with the binomial — no simulation needed!
# But we'll also simulate to confirm.

from scipy.stats import binom

lengths = [20, 30, 50, 80, 100, 150, 200, 300, 500]
n_trials = 50_000
p_match = np.sum(AA_FREQ ** 2)

results = []

print(f"p_match = Σpᵢ² = {p_match:.4f}")
print()
print(f"{'Length':>6}  {'E[id]':>7}  {'Std[id]':>8}  {'Binom p<0.05':>13}  {'Empir p<0.05':>13}  {'Binom p<0.01':>13}")
print("-" * 75)

for L in lengths:
    # Exact binomial thresholds
    binom_t05 = binom.ppf(0.95, L, p_match) / L
    binom_t01 = binom.ppf(0.99, L, p_match) / L

    # Empirical check — fully vectorized
    seqs_a = random_proteins_matrix(n_trials, L)
    seqs_b = random_proteins_matrix(n_trials, L)
    identities = np.mean(seqs_a == seqs_b, axis=1)

    emp_t05 = np.percentile(identities, 95)

    mean_id = p_match  # theoretical
    std_id = np.sqrt(p_match * (1 - p_match) / L)  # theoretical

    results.append({
        'L': L, 'mean': mean_id, 'std': std_id,
        'binom_05': binom_t05, 'binom_01': binom_t01,
        'emp_05': emp_t05
    })

    print(f"{L:>6}  {mean_id:>7.1%}  {std_id:>8.1%}  {binom_t05:>13.1%}  {emp_t05:>13.1%}  {binom_t01:>13.1%}")

print()
print("Note: binomial and empirical thresholds agree closely — the binomial model is correct!")

In [ ]:
# Visualize: significance threshold vs sequence length
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: thresholds
ax = axes[0]
Ls = [r['L'] for r in results]
ax.plot(Ls, [r['binom_05'] for r in results], 'o-', color='orange', linewidth=2, markersize=8, label='p < 0.05 (binomial)')
ax.plot(Ls, [r['binom_01'] for r in results], 's-', color='red', linewidth=2, markersize=8, label='p < 0.01 (binomial)')
ax.plot(Ls, [r['emp_05'] for r in results], 'x', color='orange', markersize=10, markeredgewidth=2, label='p < 0.05 (empirical)')
ax.axhline(p_match, color='gray', linestyle='--', linewidth=1, label=f'Random expectation ({p_match:.1%})')
ax.axhline(obs_identity, color='blue', linestyle=':', linewidth=1.5, label=f'Our observed: {obs_identity:.1%}')

ax.set_xlabel('Sequence Length', fontsize=13)
ax.set_ylabel('Percent Identity', fontsize=13)
ax.set_title('Identity Needed for Significance\nvs. Sequence Length', fontsize=14)
ax.legend(fontsize=9, loc='upper right')
ax.set_ylim(0, 0.35)
ax.grid(True, alpha=0.3)

# Right: Binomial PMFs at different lengths (proper distribution, not normal approx)
ax = axes[1]
colors = plt.cm.viridis(np.linspace(0, 0.9, len(lengths)))

for L, color in zip(lengths, colors):
    k_vals = np.arange(0, L + 1)
    pmf = binom.pmf(k_vals, L, p_match)
    # Plot as percent identity (k/L), scale density accordingly
    ax.plot(k_vals / L, pmf * L, color=color, linewidth=1.5, label=f'L={L}')

ax.axvline(obs_identity, color='red', linestyle='--', linewidth=1.5)
ax.set_xlabel('Percent Identity', fontsize=13)
ax.set_ylabel('Density', fontsize=13)
ax.set_title('Binomial Null Distributions\nNarrow with Increasing Length', fontsize=14)
ax.legend(fontsize=9, ncol=2)
ax.set_xlim(0, 0.30)

plt.tight_layout()
plt.show()

print(f"\nKey insight: at length {len(seq_a)}, our {obs_identity:.1%} identity is highly significant.")
print(f"But that same {obs_identity:.1%} at length 20 would barely clear the noise.")
print(f"At length 500, even ~{results[-1]['binom_05']:.0%} identity would be suspicious (p < 0.05).")

### What did we learn?

The **p-value** is the probability of seeing similarity *this extreme or more* under the null hypothesis (random, unrelated sequences with the same amino acid composition).

Three key insights:

1. **Each position is a coin flip.** Whether two random residues match follows a Bernoulli trial with $p = \sum p_i^2 \approx 7.3\%$, so the total number of matches follows a **Binomial distribution**. No simulation needed for the exact p-value!

2. **Our 28% identity is highly significant** at length 82 — the binomial tail probability is vanishingly small.

3. **Significance depends on length.** The binomial gets tighter (relative to its mean) as $L$ grows, because $\text{std} \propto 1/\sqrt{L}$. Short sequences need high identity to stand out; long sequences can be called significant at lower identity.

**But there's still a problem:** we treated every amino acid mismatch the same. A leucine → isoleucine swap (both hydrophobic, structurally similar) is very different biologically from leucine → aspartate (hydrophobic → charged). We need a **scoring system grounded in evolution**.

---
## Part 2 — Building a Substitution Matrix from Evolutionary Data

### The idea behind BLOSUM

The BLOSUM (BLOcks SUbstitution Matrix) captures how often amino acids substitute for each other in evolution.

Given a **Multiple Sequence Alignment (MSA)**, we:
1. Count how often each pair (a, b) appears in the same column → **observed pair frequency** $q_{ab}$
2. Count overall amino acid frequencies → **background frequencies** $p_a$
3. Compute the **log-odds score**:

$$s_{ab} = \log_2 \frac{q_{ab}}{p_a \cdot p_b}$$

This is a **log-likelihood ratio**: how much more likely is this pair under "related" vs "unrelated"?


Let's build this from a real MSA!

### Downloading a real MSA from the AlphaFold database

We'll grab the MSA for E. coli **dihydrofolate reductase (DHFR)** — a classic, well-studied enzyme. The AlphaFold DB provides precomputed MSAs in A3M format.

In [ ]:
# Download an A3M file from AlphaFold DB
# DHFR: UniProt P0ABQ4
UNIPROT_ID = "P0ABQ4"
url = f"https://alphafold.ebi.ac.uk/files/msa/AF-{UNIPROT_ID}-F1-msa_v6.a3m"

print(f"Downloading MSA for {UNIPROT_ID} (E. coli DHFR)...")
try:
    response = urllib.request.urlopen(url)
    a3m_text = response.read().decode('utf-8')
    print(f"Downloaded {len(a3m_text)} bytes")
except Exception as e:
    print(f"Download failed: {e}")
    print("Using a fallback synthetic MSA (see next cell)")
    a3m_text = None

In [ ]:
def parse_a3m(text):
    """Parse A3M format: FASTA-like but lowercase = inserts, uppercase/dash = aligned columns."""
    sequences = []
    current = []
    for line in text.strip().split('\n'):
        if line.startswith('>'):
            if current:
                sequences.append(''.join(current))
            current = []
        else:
            current.append(line.strip())
    if current:
        sequences.append(''.join(current))

    # A3M convention: remove lowercase (inserts) and '.' characters
    # Keep only uppercase letters and '-' (gaps in aligned columns)
    aligned = []
    for seq in sequences:
        clean = ''.join(c for c in seq if c.isupper() or c == '-')
        aligned.append(clean)

    return aligned

msa = parse_a3m(a3m_text)
print(f"Parsed {len(msa)} sequences, alignment length {len(msa[0])}")
# Show first few sequences (truncated)
for i, seq in enumerate(msa[:5]):
    print(f"  Seq {i}: {seq[:60]}...")

### Now let's count substitution pairs and compute the matrix

In [ ]:
def build_substitution_matrix(msa, max_seqs=2000):
    """
    Build a substitution matrix from an MSA (vectorized with NumPy).

    1. Convert MSA to integer array (20 AAs + gap code)
    2. Count observed amino acid pair frequencies (q_ab) from aligned columns
    3. Count background frequencies (p_a) from overall composition
    4. Compute log-odds: s_ab = log2(q_ab / (p_a * p_b))
    """
    n = len(AA)  # 20 amino acids
    seqs = msa[:max_seqs]
    n_seqs = len(seqs)
    aln_len = len(seqs[0])

    print(f"Using {n_seqs} sequences × {aln_len} columns")

    # Convert MSA to integer array: AA_INDEX for valid AAs, -1 for gaps/unknown
    msa_int = np.full((n_seqs, aln_len), -1, dtype=np.int8)
    for i, seq in enumerate(seqs):
        for j, c in enumerate(seq):
            if c in AA_INDEX:
                msa_int[i, j] = AA_INDEX[c]

    # --- Step 1: Count observed pair frequencies (vectorized per column) ---
    pair_counts = np.zeros((n, n), dtype=np.float64)
    single_counts = np.zeros(n, dtype=np.float64)

    for col in range(aln_len):
        residues = msa_int[:, col]
        valid = residues[residues >= 0]  # skip gaps

        if len(valid) < 2:
            continue

        # Single counts
        bc = np.bincount(valid, minlength=n)
        single_counts += bc

        # Pair counts: outer product of the column with itself
        # For each pair of sequences in this column, count (aa_i, aa_j)
        # Using bincount trick: pair_counts[a,b] += count_a * count_b (for a != b)
        # and pair_counts[a,a] += count_a * (count_a - 1) / 2 (for self-pairs)
        # Then add diagonal self-pairs
        outer = np.outer(bc, bc).astype(np.float64)
        # Subtract diagonal to avoid counting self-comparisons (seq with itself)
        np.fill_diagonal(outer, bc * (bc - 1))
        pair_counts += outer

    # Symmetrize and normalize
    total_pairs = pair_counts.sum()
    total_single = single_counts.sum()

    if total_pairs == 0 or total_single == 0:
        raise ValueError("Not enough data in MSA")

    q = pair_counts / total_pairs       # observed pair frequencies
    p = single_counts / total_single    # background frequencies

    # --- Step 2: Compute log-odds ---
    expected = np.outer(p, p)
    pseudo = 1e-10
    log_odds = np.log2((q + pseudo) / (expected + pseudo))

    # Round to half-bit integers like BLOSUM
    matrix = np.round(log_odds * 2).astype(int)

    return matrix, q, p

matrix, q_obs, p_bg = build_substitution_matrix(msa)
print(f"\nMatrix computed! Shape: {matrix.shape}")
print(f"Score range: [{matrix.min()}, {matrix.max()}]")

In [ ]:
# Dayhoff groups: amino acids ordered by biochemical similarity
# This makes the block structure of the matrix visible
DAYHOFF_ORDER = 'CSTPAGNDEQHRKMILVFYW'
DAYHOFF_GROUPS = {
    'C': 'Sulfur',
    'S': 'Small', 'T': 'Small', 'P': 'Small', 'A': 'Small', 'G': 'Small',
    'N': 'Acid/Amide', 'D': 'Acid/Amide', 'E': 'Acid/Amide', 'Q': 'Acid/Amide',
    'H': 'Basic', 'R': 'Basic', 'K': 'Basic',
    'M': 'Hydrophobic', 'I': 'Hydrophobic', 'L': 'Hydrophobic', 'V': 'Hydrophobic',
    'F': 'Aromatic', 'Y': 'Aromatic', 'W': 'Aromatic',
}
DAYHOFF_IDX = [AA_INDEX[a] for a in DAYHOFF_ORDER]

# Reorder our matrix by Dayhoff groups
matrix_dayhoff = matrix[np.ix_(DAYHOFF_IDX, DAYHOFF_IDX)]

# Group boundaries for visual separation
group_names = [DAYHOFF_GROUPS[a] for a in DAYHOFF_ORDER]
boundaries = [0]
for i in range(1, len(group_names)):
    if group_names[i] != group_names[i-1]:
        boundaries.append(i)
boundaries.append(len(DAYHOFF_ORDER))

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(matrix_dayhoff, cmap='bwr_r', vmin=-6, vmax=6, aspect='equal')

ax.set_xticks(range(len(DAYHOFF_ORDER)))
ax.set_yticks(range(len(DAYHOFF_ORDER)))
ax.set_xticklabels(list(DAYHOFF_ORDER), fontsize=11, fontfamily='monospace')
ax.set_yticklabels(list(DAYHOFF_ORDER), fontsize=11, fontfamily='monospace')

# Add score values
for i in range(20):
    for j in range(20):
        color = 'white' if abs(matrix_dayhoff[i, j]) > 3 else 'black'
        ax.text(j, i, str(matrix_dayhoff[i, j]), ha='center', va='center', fontsize=7, color=color)

# Draw group boundaries
for b in boundaries[1:-1]:
    ax.axhline(b - 0.5, color='black', linewidth=1.5)
    ax.axvline(b - 0.5, color='black', linewidth=1.5)

# Label groups on top
for k in range(len(boundaries) - 1):
    mid = (boundaries[k] + boundaries[k+1]) / 2 - 0.5
    gname = group_names[boundaries[k]]
    ax.text(mid, -1.2, gname, ha='center', va='center', fontsize=9, fontweight='bold')

ax.set_title('Our Substitution Matrix (ordered by Dayhoff groups)', fontsize=14)
plt.colorbar(im, ax=ax, label='Log-odds score', shrink=0.8)
plt.tight_layout()
plt.show()

print("Dayhoff groups cluster similar amino acids together.")
print("Notice the positive (blue) blocks along the diagonal — these are tolerated substitutions within a group.")

### Let's compare to the real BLOSUM62

Our matrix won't match exactly — BLOSUM62 uses many more MSAs and a 62% clustering threshold — but the overall pattern should be similar.

In [ ]:
# Standard BLOSUM62 matrix
BLOSUM62_STR = """
   A  C  D  E  F  G  H  I  K  L  M  N  P  Q  R  S  T  V  W  Y
A  4  0 -2 -1 -2  0 -2 -1 -1 -1 -1 -2 -1 -1 -1  1  0  0 -3 -2
C  0  9 -3 -4 -2 -3 -3 -1 -3 -1 -1 -3 -3 -3 -3 -1 -1 -1 -2 -2
D -2 -3  6  2 -3 -1 -1 -3 -1 -4 -3  1 -1  0 -2  0 -1 -3 -4 -3
E -1 -4  2  5 -3 -2  0 -3  1 -3 -2  0 -1  2  0  0 -1 -2 -3 -2
F -2 -2 -3 -3  6 -3 -1  0 -3  0  0 -3 -4 -3 -3 -2 -2 -1  1  3
G  0 -3 -1 -2 -3  6 -2 -4 -2 -4 -3  0 -2 -2 -2  0 -2 -3 -2 -3
H -2 -3 -1  0 -1 -2  8 -3 -1 -3 -2  1 -2  0  0 -1 -2 -3 -2  2
I -1 -1 -3 -3  0 -4 -3  4 -3  2  1 -3 -3 -3 -3 -2 -1  3 -3 -1
K -1 -3 -1  1 -3 -2 -1 -3  5 -2 -1  0 -1  1  2  0 -1 -2 -3 -2
L -1 -1 -4 -3  0 -4 -3  2 -2  4  2 -3 -3 -2 -2 -2 -1  1 -2 -1
M -1 -1 -3 -2  0 -3 -2  1 -1  2  5 -2 -2  0 -1 -1 -1  1 -1 -1
N -2 -3  1  0 -3  0  1 -3  0 -3 -2  6 -2  0  0  1  0 -3 -4 -2
P -1 -3 -1 -1 -4 -2 -2 -3 -1 -3 -2 -2  7 -1 -2 -1 -1 -2 -4 -3
Q -1 -3  0  2 -3 -2  0 -3  1 -2  0  0 -1  5  1  0 -1 -2 -2 -1
R -1 -3 -2  0 -3 -2  0 -3  2 -2 -1  0 -2  1  5 -1 -1 -3 -3 -2
S  1 -1  0  0 -2  0 -1 -2  0 -2 -1  1 -1  0 -1  4  1 -2 -3 -2
T  0 -1 -1 -1 -2 -2 -2 -1 -1 -1 -1  0 -1 -1 -1  1  5  0 -2 -2
V  0 -1 -3 -2 -1 -3 -3  3 -2  1  1 -3 -2 -2 -3 -2  0  4 -3 -1
W -3 -2 -4 -3  1 -2 -2 -3 -3 -2 -1 -4 -4 -2 -3 -3 -2 -3 11  2
Y -2 -2 -3 -2  3 -3  2 -1 -2 -1 -1 -2 -3 -1 -2 -2 -2 -1  2  7
"""

def parse_blosum62(text):
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    header = lines[0].split()
    mat = np.zeros((20, 20), dtype=int)
    for i, line in enumerate(lines[1:]):
        values = line.split()
        for j, v in enumerate(values[1:]):
            mat[i, j] = int(v)
    return mat, header

blosum62, blosum_aa = parse_blosum62(BLOSUM62_STR)

# Reindex to match our AA order
blosum_reindexed = np.zeros((20, 20), dtype=int)
blosum_idx = {a: i for i, a in enumerate(blosum_aa)}
for i, a in enumerate(AA):
    for j, b in enumerate(AA):
        blosum_reindexed[i, j] = blosum62[blosum_idx[a], blosum_idx[b]]

print("BLOSUM62 loaded and reindexed to our amino acid order.")

In [ ]:
# Side-by-side comparison — both in Dayhoff order
blosum_dayhoff = blosum_reindexed[np.ix_(DAYHOFF_IDX, DAYHOFF_IDX)]

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for ax, mat, title in [(axes[0], matrix_dayhoff, 'Our Matrix (from MSA)'),
                         (axes[1], blosum_dayhoff, 'BLOSUM62')]:
    im = ax.imshow(mat, cmap='bwr_r', vmin=-6, vmax=6, aspect='equal')
    ax.set_xticks(range(len(DAYHOFF_ORDER)))
    ax.set_yticks(range(len(DAYHOFF_ORDER)))
    ax.set_xticklabels(list(DAYHOFF_ORDER), fontsize=9, fontfamily='monospace')
    ax.set_yticklabels(list(DAYHOFF_ORDER), fontsize=9, fontfamily='monospace')
    ax.set_title(title, fontsize=14)
    # Draw group boundaries
    for b in boundaries[1:-1]:
        ax.axhline(b - 0.5, color='black', linewidth=1)
        ax.axvline(b - 0.5, color='black', linewidth=1)
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Substitution Matrix Comparison (Dayhoff ordering)', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: our scores vs BLOSUM62 (element-by-element)
# Use upper triangle only (matrix is symmetric) to avoid double-counting
our_flat = []
bl_flat = []
labels = []
colors_scatter = []

# Color by Dayhoff group relationship
group_colors = {
    'same': '#2196F3',      # blue — same AA
    'within': '#4CAF50',    # green — same Dayhoff group
    'between': '#9E9E9E',   # gray — different groups
}

for i in range(20):
    for j in range(i, 20):  # upper triangle including diagonal
        our_flat.append(matrix[i, j])
        bl_flat.append(blosum_reindexed[i, j])
        aa_i, aa_j = AA[i], AA[j]
        labels.append(f"{aa_i}{aa_j}")
        if i == j:
            colors_scatter.append(group_colors['same'])
        elif DAYHOFF_GROUPS[aa_i] == DAYHOFF_GROUPS[aa_j]:
            colors_scatter.append(group_colors['within'])
        else:
            colors_scatter.append(group_colors['between'])

our_flat = np.array(our_flat)
bl_flat = np.array(bl_flat)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(bl_flat, our_flat, c=colors_scatter, s=40, alpha=0.7, edgecolors='white', linewidth=0.5)

# Add diagonal reference line
lims = [min(bl_flat.min(), our_flat.min()) - 1, max(bl_flat.max(), our_flat.max()) + 1]
ax.plot(lims, lims, 'k--', alpha=0.3, linewidth=1)

# Label a few interesting points (diagonal elements = self-substitution scores)
for i in range(20):
    idx = [k for k, l in enumerate(labels) if l == AA[i]*2][0] if AA[i]*2 in labels else None
    if idx is not None and (abs(our_flat[idx]) > 3 or abs(bl_flat[idx]) > 5):
        ax.annotate(AA[i], (bl_flat[idx], our_flat[idx]), fontsize=9, fontweight='bold',
                    xytext=(5, 5), textcoords='offset points')

corr = np.corrcoef(our_flat, bl_flat)[0, 1]

# Legend
import matplotlib.patches as mpatches
legend_handles = [
    mpatches.Patch(color=group_colors['same'], label='Same AA (diagonal)'),
    mpatches.Patch(color=group_colors['within'], label='Within Dayhoff group'),
    mpatches.Patch(color=group_colors['between'], label='Between groups'),
]
ax.legend(handles=legend_handles, fontsize=10, loc='upper left')

ax.set_xlabel('BLOSUM62 score', fontsize=13)
ax.set_ylabel('Our matrix score', fontsize=13)
ax.set_title(f'Our Matrix vs BLOSUM62 (r = {corr:.3f})', fontsize=14)
ax.set_aspect('equal')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print(f"Pearson correlation: {corr:.3f}")
print(f"Our matrix was built from a SINGLE protein family's MSA.")
print(f"BLOSUM62 was built from thousands of diverse alignment blocks.")
print(f"The correlation shows we've captured the fundamental biochemistry!")

### Key observations

- **Block structure is visible** in the Dayhoff ordering: similar amino acids cluster together with positive (red) scores
- **Hydrophobic block** (M, I, L, V): large positive region — these substitute freely in protein cores
- **Aromatic block** (F, Y, W): another positive cluster, especially F↔Y
- **Acid/amide block** (N, D, E, Q): charge-similar residues tolerate substitution
- **Cysteine (C)** has the highest self-score: disulfide bonds demand conservation
- **Cross-group substitutions** (e.g. hydrophobic ↔ charged) are strongly negative

The scatter plot shows our single-MSA matrix correlates well with BLOSUM62, which was built from thousands of alignment blocks. The underlying biochemistry is robust!

The score $s_{ab}$ is literally **bits of evidence** for homology at each position. Now we need to use this for alignment.

---
## Part 3 — Smith-Waterman Local Alignment

In real life, we don't have pre-aligned sequences. We need to **find** the best local alignment, allowing gaps (insertions/deletions).

### The recurrence

Smith-Waterman fills a matrix $H$ where $H[i,j]$ is the best local alignment score ending at position $i$ in sequence 1 and $j$ in sequence 2:

$$H[i,j] = \max \begin{cases} 0 \\ H[i-1, j-1] + s(a_i, b_j) & \text{(match/mismatch)} \\ H[i-1, j] - d & \text{(gap in seq 2)} \\ H[i, j-1] - d & \text{(gap in seq 1)} \end{cases}$$

where $s(a_i, b_j)$ is the substitution matrix score and $d$ is the gap penalty.

The key difference from Needleman-Wunsch (global alignment): the **zero floor** — alignment can restart anywhere, giving us local matches.

In [ ]:
def smith_waterman(seq1, seq2, subst_matrix, aa_index, gap_penalty=10):
    """
    Smith-Waterman local alignment (optimized with precomputed lookups).

    Parameters:
        seq1, seq2:      amino acid sequences (strings)
        subst_matrix:    20x20 numpy array of substitution scores
        aa_index:        dict mapping amino acid -> index
        gap_penalty:     linear gap penalty (positive number, will be subtracted)

    Returns:
        score:      best local alignment score
        alignment:  tuple of (aligned_seq1, aligned_seq2, midline)
        H:          the full scoring matrix (for visualization)
    """
    m, n = len(seq1), len(seq2)

    # Precompute integer indices for both sequences (avoids dict lookups in inner loop)
    idx1 = np.array([aa_index.get(c, -1) for c in seq1], dtype=np.int8)
    idx2 = np.array([aa_index.get(c, -1) for c in seq2], dtype=np.int8)

    # Precompute the full score row for each position in seq1 vs all of seq2
    # scores[i, j] = subst_matrix[seq1[i], seq2[j]]
    # Build extended matrix with row/col for unknown AAs (index -1 → penalty)
    ext_matrix = np.full((21, 21), -4, dtype=np.float64)
    ext_matrix[:20, :20] = subst_matrix
    scores = ext_matrix[idx1][:, idx2]  # shape (m, n)

    # Initialize scoring matrix and traceback
    H = np.zeros((m + 1, n + 1), dtype=np.float64)
    traceback = np.zeros((m + 1, n + 1), dtype=np.int8)
    # Traceback codes: 0=stop, 1=diagonal, 2=up, 3=left

    best_score = 0
    best_pos = (0, 0)

    # Fill the matrix — the DP recurrence can't be fully vectorized,
    # but we can vectorize each row (all j values for a given i)
    for i in range(1, m + 1):
        diag  = H[i-1, :-1] + scores[i-1]      # match/mismatch
        up    = H[i-1, 1:]  - gap_penalty        # gap in seq2
        left  = H[i, :-1]   - gap_penalty        # gap in seq1 — BUT this depends on H[i] being filled left-to-right

        # Unfortunately, 'left' creates a dependency chain (H[i,j] depends on H[i,j-1])
        # so we can't fully vectorize. We vectorize diag and up, then loop for left.
        candidates = np.maximum(0, np.maximum(diag, up))

        for j in range(1, n + 1):
            left_val = H[i, j-1] - gap_penalty
            H[i, j] = max(candidates[j-1], left_val)

            if H[i, j] == 0:
                traceback[i, j] = 0
            elif H[i, j] == diag[j-1]:
                traceback[i, j] = 1
            elif H[i, j] == up[j-1]:
                traceback[i, j] = 2
            else:
                traceback[i, j] = 3

            if H[i, j] > best_score:
                best_score = H[i, j]
                best_pos = (i, j)

    # Traceback from best score
    aligned1, aligned2, midline = [], [], []
    i, j = best_pos

    while i > 0 and j > 0 and H[i, j] > 0:
        if traceback[i, j] == 1:  # diagonal
            a, b = seq1[i-1], seq2[j-1]
            aligned1.append(a)
            aligned2.append(b)
            midline.append('|' if a == b else ('+' if ext_matrix[idx1[i-1], idx2[j-1]] > 0 else '.'))
            i -= 1
            j -= 1
        elif traceback[i, j] == 2:  # up
            aligned1.append(seq1[i-1])
            aligned2.append('-')
            midline.append(' ')
            i -= 1
        elif traceback[i, j] == 3:  # left
            aligned1.append('-')
            aligned2.append(seq2[j-1])
            midline.append(' ')
            j -= 1
        else:
            break

    # Reverse (we traced backwards)
    aligned1 = ''.join(reversed(aligned1))
    aligned2 = ''.join(reversed(aligned2))
    midline = ''.join(reversed(midline))

    return best_score, (aligned1, aligned2, midline), H

def sw_score_only(seq1, seq2, subst_matrix, aa_index, gap_penalty=10):
    """Smith-Waterman returning only the best score (no traceback — faster for database scanning)."""
    m, n = len(seq1), len(seq2)
    idx1 = np.array([aa_index.get(c, -1) for c in seq1], dtype=np.int8)
    idx2 = np.array([aa_index.get(c, -1) for c in seq2], dtype=np.int8)

    ext_matrix = np.full((21, 21), -4, dtype=np.float64)
    ext_matrix[:20, :20] = subst_matrix
    scores = ext_matrix[idx1][:, idx2]

    # Only keep two rows (current and previous) — O(n) memory
    prev = np.zeros(n + 1, dtype=np.float64)
    best = 0.0

    for i in range(m):
        curr = np.zeros(n + 1, dtype=np.float64)
        diag = prev[:-1] + scores[i]     # diagonal scores for all j
        up   = prev[1:]  - gap_penalty    # gap in seq2 for all j

        for j in range(n):
            left = curr[j] - gap_penalty
            curr[j+1] = max(0, diag[j], up[j], left)

        best = max(best, curr.max())
        prev = curr

    return best

print("Smith-Waterman functions defined!")
print("  smith_waterman() — full alignment with traceback (for visualization)")
print("  sw_score_only()  — score only, no traceback (faster, for database search)")

### Let's align two real sequences

We'll use our BLOSUM62 matrix for scoring (it's more robust than our single-MSA matrix for general use).

In [ ]:
# Two E. coli proteins: RecA (DNA repair ATPase) and RadA (distant homolog)
# Different lengths, moderate similarity — SW should find the conserved ATPase core
query =  "AIDENKQKALAAALGQIEKQFGKGSIMRLGEDRSMDVETISTGSLSLDIALGAGGLPMGRIVEIYGPESSGKTTLTLQVIAAAQREGKTCAFIDAEHALDPIYARKLGVDIDNLLCSQPDTGEQALEICDALARSGAVDVIVVDSVAALTPKAEIE"
target = "MSDQFIEQAFNALPGIEKSYGKGSSLRLADEGRSTGETISISGRLALETALQAGGLRGRISLYGTESSSGKTSVALQAIAAAQKTTGTCAFIDAEHALDPTYARQLGVIDIDDKLSKRPDKAGEQALEIFSYALERGDIDLIVVDSVAALTPKAEIE"

print(f"Query  ({len(query)} aa):  {query[:60]}...")
print(f"Target ({len(target)} aa): {target[:60]}...")
print()

t0 = time.time()
score, alignment, H = smith_waterman(query, target, blosum_reindexed, AA_INDEX, gap_penalty=10)
elapsed = time.time() - t0

aln1, aln2, mid = alignment

print(f"Score: {score}")
print(f"Time:  {elapsed:.3f}s")
print()
print("Alignment:")
# Print alignment in blocks of 60
for start in range(0, len(aln1), 60):
    print(f"  Query:  {aln1[start:start+60]}")
    print(f"          {mid[start:start+60]}")
    print(f"  Target: {aln2[start:start+60]}")
    print()

In [ ]:
# Visualize the DP matrix
fig, ax = plt.subplots(figsize=(14, 10))
im = ax.imshow(H[1:, 1:], cmap='YlOrRd', aspect='auto')

# Label axes with sequence residues
step_x = max(1, len(target) // 40)
step_y = max(1, len(query) // 40)
ax.set_xticks(range(0, len(target), step_x))
ax.set_xticklabels([target[i] for i in range(0, len(target), step_x)], fontsize=8, fontfamily='monospace')
ax.set_yticks(range(0, len(query), step_y))
ax.set_yticklabels([query[i] for i in range(0, len(query), step_y)], fontsize=8, fontfamily='monospace')

ax.set_xlabel('Target sequence', fontsize=12)
ax.set_ylabel('Query sequence', fontsize=12)
ax.set_title(f'Smith-Waterman DP Matrix (best score = {score})', fontsize=14)
plt.colorbar(im, ax=ax, label='Alignment score', shrink=0.8)
plt.tight_layout()
plt.show()

### What does the score mean?

The alignment score is a **sum of log-odds** across the aligned positions:

$$\text{Score} = \sum_{\text{aligned pairs}} s(a_i, b_j) - \sum_{\text{gaps}} d$$

So the total score is the **total log-likelihood ratio** — how much more likely is this alignment under "homologous" vs "random"?

But again — **is this score significant?** We need a p-value.

---
## Part 4 — P-values Revisited: Now With Alignment Scores

This time, instead of measuring identity on pre-aligned sequences, we'll ask:

> "If I shuffle the target and re-run Smith-Waterman, what scores do I get by chance?"

This null distribution accounts for:
- The substitution matrix (evolutionary information)
- Gap handling
- The "best local match" effect (optimistic by design)

In [ ]:
# Build null distribution of Smith-Waterman scores
# (This takes a minute — we need many alignments)

n_shuffles = 500  # each one runs full SW, so we keep this manageable
target_arr = np.array(list(target))

null_scores = np.zeros(n_shuffles)
print(f"Running {n_shuffles} shuffled alignments...")

t0 = time.time()
for i in range(n_shuffles):
    shuffled = target_arr.copy()
    np.random.shuffle(shuffled)
    null_scores[i] = sw_score_only(query, ''.join(shuffled), blosum_reindexed, AA_INDEX, gap_penalty=10)
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{n_shuffles} done ({time.time()-t0:.1f}s)")

elapsed = time.time() - t0
print(f"Done in {elapsed:.1f}s")

p_value_sw = np.mean(null_scores >= score)
print(f"\nObserved score: {score}")
print(f"Null mean ± std: {null_scores.mean():.1f} ± {null_scores.std():.1f}")
print(f"Empirical p-value: {p_value_sw}")

In [ ]:
# Visualize and fit an Extreme Value Distribution
from scipy.stats import gumbel_r  # this is in numpy ecosystem, standard in colab

fig, ax = plt.subplots(figsize=(10, 5))

# Histogram of null scores
ax.hist(null_scores, bins=40, density=True, alpha=0.7, color='steelblue', edgecolor='white', label='Shuffled scores')

# Fit Gumbel (Extreme Value Distribution) — this is what Karlin-Altschul theory predicts
try:
    from scipy.stats import gumbel_r
    loc, scale = gumbel_r.fit(null_scores)
    x = np.linspace(null_scores.min() - 5, max(null_scores.max(), score) + 10, 200)
    ax.plot(x, gumbel_r.pdf(x, loc=loc, scale=scale), 'k-', linewidth=2, label=f'Gumbel fit (μ={loc:.1f}, β={scale:.1f})')

    # Analytical p-value from EVD
    p_value_evd = 1 - gumbel_r.cdf(score, loc=loc, scale=scale)
    ax.text(0.95, 0.95, f'EVD p-value: {p_value_evd:.2e}', transform=ax.transAxes,
            fontsize=12, ha='right', va='top', bbox=dict(boxstyle='round', facecolor='wheat'))
except ImportError:
    print("scipy not available — showing empirical distribution only")

ax.axvline(score, color='red', linewidth=2, linestyle='--', label=f'Observed: {score}')
ax.set_xlabel('Smith-Waterman Score', fontsize=13)
ax.set_ylabel('Density', fontsize=13)
ax.set_title('Null Distribution of SW Scores\n(Karlin-Altschul: scores follow an Extreme Value Distribution)', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### Why an Extreme Value Distribution?

When you take the **maximum** of many random variables (which is what Smith-Waterman does — it finds the best local match), the result follows a **Gumbel distribution** (a type of EVD). This is a fundamental result from extreme value theory.

**Karlin & Altschul (1990)** proved this for ungapped alignments, and it holds approximately for gapped alignments too. This is the theoretical foundation of BLAST's statistics.

The key formula: for a score $S$, the probability of seeing it by chance is approximately:

$$P(S \geq x) \approx 1 - \exp(-K \cdot m \cdot n \cdot e^{-\lambda x})$$

where $m, n$ are sequence lengths and $K, \lambda$ are parameters estimated from the score distribution.

---
## Part 5 — Searching a Database: E-values and Multiple Testing

Now the real scenario: you have a query protein and want to find its homologs in an entire genome.

### The problem with p-values in database searches

If you search against 4,300 E. coli proteins with a p-value threshold of 0.001, you expect **4.3 false positives** just by chance. The p-value alone is misleading when you do thousands of comparisons.

### Enter the E-value

$$E = p \times D$$

where $D$ is the database size (number of sequences searched).

**E-value = expected number of hits this good or better by chance in a database of this size.**

- E = 0.001 → you'd need to search ~1,000 databases to see one false positive this good
- E = 10 → you'd expect 10 hits this good even with random sequences

Let's see this in action.

In [ ]:
# Download E. coli K-12 proteome from UniProt
ECOLI_URL = "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=organism_id:83333+AND+reviewed:true"

print("Downloading E. coli K-12 proteome from UniProt...")
try:
    req = urllib.request.Request(ECOLI_URL, headers={'User-Agent': 'Python/tutorial'})
    response = urllib.request.urlopen(req, timeout=30)
    fasta_text = response.read().decode('utf-8')
    print(f"Downloaded {len(fasta_text)} bytes")
except Exception as e:
    print(f"Download failed: {e}")
    print("Generating synthetic E. coli-like proteome...")
    fasta_text = None

In [ ]:
def parse_fasta(text):
    """Parse FASTA format into list of (header, sequence) tuples."""
    sequences = []
    header = ""
    current = []
    for line in text.strip().split('\n'):
        if line.startswith('>'):
            if current:
                sequences.append((header, ''.join(current)))
            header = line[1:].split()[0]
            current = []
        else:
            current.append(line.strip())
    if current:
        sequences.append((header, ''.join(current)))
    return sequences

if fasta_text:
    ecoli_proteins = parse_fasta(fasta_text)
else:
    # Generate synthetic proteome as fallback
    print("Generating synthetic proteome (500 proteins)...")
    aa_freq = [0.087, 0.033, 0.047, 0.050, 0.040, 0.089, 0.034, 0.037, 0.081, 0.085,
               0.015, 0.040, 0.051, 0.038, 0.041, 0.070, 0.058, 0.065, 0.010, 0.030]
    ecoli_proteins = []
    for k in range(500):
        length = np.random.randint(80, 500)
        seq = ''.join(np.random.choice(list(AA), size=length, p=aa_freq))
        ecoli_proteins.append((f"synthetic_{k:04d}", seq))

print(f"Loaded {len(ecoli_proteins)} E. coli proteins")
# Show length distribution
lengths = [len(s) for _, s in ecoli_proteins]
print(f"Length range: {min(lengths)}-{max(lengths)} aa (median {np.median(lengths):.0f})")

### Experiment 1: Search with a random query (expect no real hits)

In [ ]:
# Generate a random query protein
random_query = random_protein(100)
print(f"Random query ({len(random_query)} aa): {random_query[:60]}...")

# Search against a subset (full proteome would be slow with pure Python SW)
#n_targets = min(500, len(ecoli_proteins))
subset = []
for p in ecoli_proteins:
  if len(p[1]) <= 512:
      subset.append(p)
#subset = ecoli_proteins #[ecoli_proteins[i] for i in np.random.choice(len(ecoli_proteins), n_targets, replace=False)]
n_targets = len(subset)

print(f"\nSearching against {n_targets} E. coli proteins...")
random_scores = []
t0 = time.time()

for i, (name, target_seq) in enumerate(subset):
    s = sw_score_only(random_query, target_seq[:200], blosum_reindexed, AA_INDEX, gap_penalty=10)
    random_scores.append((s, name))
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{n_targets} ({time.time()-t0:.1f}s)")

random_scores.sort(reverse=True)
elapsed = time.time() - t0
print(f"Done in {elapsed:.1f}s")

print(f"\nTop 5 hits (random query):")
for score_val, name in random_scores[:5]:
    print(f"  {name}: score = {score_val:.0f}")

### Experiment 2: Search with a real query (RecA — expect to find homologs)

In [ ]:
# Use RecA as query — a well-conserved protein that should have homologs
dhfr_query = "AIDENKQKALAAALGQIEKQFGKGSIMRLGEDRSMDVETISTGSLSLDIALGAGGLPMGRIVEIYGPESSGKTTLTLQVIAAAQREGKTCAFIDAEHALDPIYARKLGVDIDNLLCSQPDTGEQALEICDALARSGAVDVIVVDSVAALTPKAEIE"
print(f"RecA query ({len(dhfr_query)} aa): {dhfr_query[:60]}...")

print(f"\nSearching against {n_targets} E. coli proteins...")
real_scores = []
t0 = time.time()

for i, (name, target_seq) in enumerate(subset):
    s = sw_score_only(dhfr_query, target_seq[:200], blosum_reindexed, AA_INDEX, gap_penalty=10)
    real_scores.append((s, name))
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{n_targets} ({time.time()-t0:.1f}s)")

real_scores.sort(reverse=True)
elapsed = time.time() - t0
print(f"Done in {elapsed:.1f}s")

print(f"\nTop 10 hits (RecA query):")
for score_val, name in real_scores[:10]:
    print(f"  {name}: score = {score_val}")

In [ ]:
# Compare score distributions and compute E-values
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, scores_list, title, color in [
    (axes[0], [s for s, _ in random_scores], 'Random Query', 'steelblue'),
    (axes[1], [s for s, _ in real_scores], 'RecA Query (has homologs)', 'darkgreen')
]:
    scores_arr = np.array(scores_list)
    ax.hist(scores_arr, bins=30, density=True, alpha=0.7, color=color, edgecolor='white')

    # Fit EVD to the bulk of the distribution (background)
    try:
        from scipy.stats import gumbel_r
        # Use the lower 90% of scores for fitting (exclude potential true hits)
        threshold = np.percentile(scores_arr, 90)
        bg_scores = scores_arr[scores_arr <= threshold]
        loc, scale = gumbel_r.fit(bg_scores)
        x = np.linspace(scores_arr.min() - 5, scores_arr.max() + 10, 200)
        ax.plot(x, gumbel_r.pdf(x, loc=loc, scale=scale), 'k-', linewidth=2, label='EVD fit (background)')
    except:
        pass

    ax.set_xlabel('Smith-Waterman Score', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.legend(fontsize=10)

plt.suptitle('Database Search: Score Distributions', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# Compute E-values for the RecA search
print("E-value computation for RecA search")
print("=" * 55)
print()

try:
    from scipy.stats import gumbel_r

    all_scores = np.array([s for s, _ in real_scores])

    # Fit EVD to background (lower 90%)
    threshold = np.percentile(all_scores, 90)
    bg = all_scores[all_scores <= threshold]
    loc, scale = gumbel_r.fit(bg)

    D = n_targets  # database size

    print(f"Database size: {D}")
    print(f"EVD parameters: μ = {loc:.2f}, β = {scale:.2f}")
    print()
    print(f"{'Rank':<6} {'Protein':<20} {'Score':<8} {'P-value':<12} {'E-value':<12} {'Significant?'}")
    print("-" * 75)

    for rank, (score_val, name) in enumerate(real_scores[:15], 1):
        p_val = 1 - gumbel_r.cdf(score_val, loc=loc, scale=scale)
        e_val = p_val * D
        sig = "***" if e_val < 0.001 else "**" if e_val < 0.01 else "*" if e_val < 1 else ""
        print(f"{rank:<6} {name:<20} {score_val:<8.0f} {p_val:<12.2e} {e_val:<12.4f} {sig}")

    print()
    print("Significance: *** E < 0.001,  ** E < 0.01,  * E < 1")

except ImportError:
    print("scipy needed for EVD fitting — install with: pip install scipy")

### Understanding E-values

| E-value | Meaning |
|---------|---------|
| 0.001 | Expect this by chance once in 1,000 database searches → **very confident** |
| 0.01 | Once in 100 searches → **confident** |
| 1 | Expect 1 false positive per search → **borderline** |
| 10 | Expect 10 hits this good by chance → **not significant** |

The E-value is essentially a **Bonferroni correction**: we multiply the p-value by the number of tests.

$$E = p \times D$$

BLAST uses a more sophisticated version that accounts for database size and sequence lengths in the EVD parameters themselves, but the core idea is identical.

---
## Summary: The Full Pipeline

We've built, from scratch, the core statistical framework behind tools like BLAST:

1. **Percent identity** → too naive, treats all substitutions equally
2. **Substitution matrix** → log-odds scores from evolutionary data (BLOSUM)
3. **Smith-Waterman** → finds optimal local alignment using the matrix
4. **P-value** → "is this score surprising?" via null distribution (shuffled sequences)
5. **E-value** → "is this score surprising *given how many comparisons I did*?" (multiple testing correction)

### What BLAST adds on top of this:
- **Heuristic seeding** — instead of full Smith-Waterman against every database entry, find short exact matches (seeds) first, then extend only promising ones. This is ~1000× faster.
- **Pre-computed EVD parameters** — avoids the expensive shuffling step
- **Gapped extension** — uses banded dynamic programming for speed

But the statistical foundations are exactly what we built here.

### Key equations to remember:

**Log-odds score:** $s_{ab} = \log_2 \frac{q_{ab}}{p_a \cdot p_b}$ (each position contributes evidence for/against homology)

**Alignment score:** $S = \sum s(a_i, b_j) - \text{gap penalties}$ (total evidence across the alignment)

**E-value:** $E = p \times D$ (expected false positives at this score threshold)

---
*Tutorial built for demonstration purposes. For production bioinformatics, use established tools like BLAST, HMMER, MMseqs2, etc.*